In [2]:
ratings = {
    "Alice":   {"Inception": 5, "Titanic": 3, "Avatar": 4, "Joker": 2, "Interstellar": 5, "Frozen": 1},
    "Bob":     {"Inception": 4, "Titanic": 1, "Avatar": 5, "Joker": 3, "Interstellar": 4, "Frozen": 2},
    "Charlie": {"Inception": 2, "Titanic": 5, "Avatar": 1, "Joker": 4, "Interstellar": 2, "Frozen": 5},
    "Diana":   {"Inception": 5, "Titanic": 2, "Avatar": 4, "Joker": 1, "Interstellar": 5, "Frozen": 2},
    "Eve":     {"Inception": 1, "Titanic": 5, "Avatar": 2, "Joker": 5, "Interstellar": 1, "Frozen": 4},
    "Frank":   {"Inception": 4, "Titanic": 0, "Avatar": 3, "Joker": 0, "Interstellar": 5, "Frozen": 0},
}


all_movies = list(ratings["Alice"].keys())

print("=" * 45)
print("   MOVIE RECOMMENDATION SYSTEM")
print("=" * 45)
print(f"\nMovies in system : {all_movies}")
print(f"Users in system  : {list(ratings.keys())}")


   MOVIE RECOMMENDATION SYSTEM

Movies in system : ['Inception', 'Titanic', 'Avatar', 'Joker', 'Interstellar', 'Frozen']
Users in system  : ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 'Frank']


In [3]:
print("\nAlice's ratings:")
for movie, score in ratings["Bob"].items():
    stars = "⭐" * score if score > 0 else "not watched"
    print(f"  {movie:<15} {stars}")


Alice's ratings:
  Inception       ⭐⭐⭐⭐
  Titanic         ⭐
  Avatar          ⭐⭐⭐⭐⭐
  Joker           ⭐⭐⭐
  Interstellar    ⭐⭐⭐⭐
  Frozen          ⭐⭐


In [4]:
import math  # for math.sqrt()

def cosine_similarity(user1_ratings, user2_ratings):
    """
    Calculate how similar two users are based on their ratings.
    Returns a score from 0 (totally different) to 1 (identical taste).
    """
    # Only compare movies BOTH users have rated (no zeros)
    common_movies = [
        movie for movie in user1_ratings
        if user1_ratings[movie] > 0 and user2_ratings[movie] > 0
    ]

    # If they share no movies in common, similarity = 0
    if len(common_movies) == 0:
        return 0

    # Step 1: Dot product — multiply matching ratings and add them up
    dot_product = sum(
        user1_ratings[m] * user2_ratings[m]
        for m in common_movies
    )

    # Step 2: Magnitude of each user's ratings vector
    magnitude1 = math.sqrt(sum(user1_ratings[m] ** 2 for m in common_movies))
    magnitude2 = math.sqrt(sum(user2_ratings[m] ** 2 for m in common_movies))

    # Step 3: Divide dot product by both magnitudes
    if magnitude1 == 0 or magnitude2 == 0:
        return 0

    return dot_product / (magnitude1 * magnitude2)


# ✅ TEST IT — compare users
print("Similarity scores with Alice:\n")
for user in ratings:
    if user == "Alice":
        continue
    score = cosine_similarity(ratings["Alice"], ratings[user])
    bar   = "█" * int(score * 20)   # visual bar
    print(f"  Alice vs {user:<8} {score:.3f}  {bar}")

print("\n💡 Higher score = more similar taste to Alice")
print("   Diana is most similar (both love sci-fi)")
print("   Eve is most different (loves what Alice hates)")

Similarity scores with Alice:

  Alice vs Bob      0.942  ██████████████████
  Alice vs Charlie  0.671  █████████████
  Alice vs Diana    0.981  ███████████████████
  Alice vs Eve      0.619  ████████████
  Alice vs Frank    0.992  ███████████████████

💡 Higher score = more similar taste to Alice
   Diana is most similar (both love sci-fi)
   Eve is most different (loves what Alice hates)


In [5]:

def find_similar_users(target_user, all_ratings, top_n=3):

    similarities = []

    for user in all_ratings:
        if user == target_user:
            continue   # skip comparing user to themselves

        score = cosine_similarity(
            all_ratings[target_user],
            all_ratings[user]
        )
        similarities.append((user, score))

    # CONCEPT: sorted() with a key function + reverse=True for descending
    # key=lambda x: x[1] means "sort by the second item in each tuple"
    similarities = sorted(similarities, key=lambda x: x[1], reverse=True)

    return similarities[:top_n]   # return only top N


def recommend_movies(target_user, all_ratings, top_n_users=3, top_n_movies=3):
    """
    Recommend movies to target_user using collaborative filtering.
    Returns a list of (movie, predicted_score) tuples.
    """
    # Step 1: Find movies the user has NOT watched yet
    unwatched = [
        movie for movie in all_movies
        if all_ratings[target_user][movie] == 0
    ]

    if not unwatched:
        return []   # user has watched everything

    # Step 2: Find similar users
    similar_users = find_similar_users(target_user, all_ratings, top_n_users)

    # Step 3: For each unwatched movie, calculate a predicted score
    # Predicted score = weighted average of similar users' ratings
    # Weight = similarity score (more similar user = more influence)
    predictions = []

    for movie in unwatched:
        weighted_total  = 0   # sum of (similarity × rating)
        similarity_total = 0   # sum of similarities (for averaging)

        for (similar_user, similarity) in similar_users:
            their_rating = all_ratings[similar_user][movie]

            if their_rating > 0:   # only count if they've seen it
                weighted_total   += similarity * their_rating
                similarity_total += similarity

        if similarity_total > 0:
            predicted_score = weighted_total / similarity_total
            predictions.append((movie, predicted_score))

    # Step 4: Sort by predicted score, highest first
    predictions = sorted(predictions, key=lambda x: x[1], reverse=True)

    return predictions[:top_n_movies]


# ✅ TEST IT
print("Finding similar users to Frank...\n")
similar = find_similar_users("Frank", ratings)
for user, score in similar:
    print(f"  {user:<10} similarity: {score:.3f}  {'█' * int(score * 20)}")

print("\nRecommendations for Frank (hasn't seen Titanic, Joker, Frozen):\n")
recs = recommend_movies("Frank", ratings)
for i, (movie, score) in enumerate(recs, 1):
    stars = "⭐" * round(score)
    print(f"  {i}. {movie:<15} predicted rating: {score:.2f}  {stars}")

Finding similar users to Frank...

  Alice      similarity: 0.992  ███████████████████
  Diana      similarity: 0.992  ███████████████████
  Charlie    similarity: 0.990  ███████████████████

Recommendations for Frank (hasn't seen Titanic, Joker, Frozen):

  1. Titanic         predicted rating: 3.33  ⭐⭐⭐
  2. Frozen          predicted rating: 2.66  ⭐⭐⭐
  3. Joker           predicted rating: 2.33  ⭐⭐


In [6]:

movie_features = {
    "Inception":    ["sci-fi", "thriller", "mind-bending"],
    "Titanic":      ["romance", "drama", "historical"],
    "Avatar":       ["sci-fi", "action", "fantasy"],
    "Joker":        ["thriller", "drama", "crime"],
    "Interstellar": ["sci-fi", "drama", "mind-bending"],
    "Frozen":       ["animation", "fantasy", "romance"],
}

def build_user_profile(user, all_ratings, features):
    """
    Build a profile of genres this user likes, based on their ratings.
    High ratings → those genres get a high weight in the profile.
    """
    # CONCEPT: defaultdict-style manual counting using a regular dict
    genre_scores = {}

    for movie, rating in all_ratings[user].items():
        if rating == 0:
            continue   # skip unwatched movies

        for genre in features[movie]:
            if genre not in genre_scores:
                genre_scores[genre] = 0
            # Add the rating to that genre's score
            genre_scores[genre] += rating

    return genre_scores


def content_recommend(user, all_ratings, features, top_n=3):
    """
    Recommend unwatched movies based on genres the user likes.
    """
    # Step 1: Build the user's genre preference profile
    profile = build_user_profile(user, all_ratings, features)

    # Step 2: Find unwatched movies
    unwatched = [m for m in all_movies if all_ratings[user][m] == 0]

    # Step 3: Score each unwatched movie based on its genre match
    scores = []
    for movie in unwatched:
        match_score = 0
        for genre in features[movie]:
            # Add the user's preference score for each genre in this movie
            match_score += profile.get(genre, 0)
        scores.append((movie, match_score))

    # Step 4: Sort by best match
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    return scores[:top_n]


# ✅ TEST IT
print("Alice's genre profile (what she likes):\n")
alice_profile = build_user_profile("Alice", ratings, movie_features)
for genre, score in sorted(alice_profile.items(), key=lambda x: x[1], reverse=True):
    bar = "█" * int(score / 2)
    print(f"  {genre:<15} score: {score}  {bar}")

print("\nContent-based recommendations for Alice:\n")
recs = content_recommend("Alice", ratings, movie_features)
for i, (movie, score) in enumerate(recs, 1):
    print(f"  {i}. {movie:<15} genre match score: {score}")

Alice's genre profile (what she likes):

  sci-fi          score: 14  ███████
  mind-bending    score: 10  █████
  drama           score: 10  █████
  thriller        score: 7  ███
  fantasy         score: 5  ██
  romance         score: 4  ██
  action          score: 4  ██
  historical      score: 3  █
  crime           score: 2  █
  animation       score: 1  

Content-based recommendations for Alice:



In [ ]:
# CONCEPT: Combining everything into one interactive program
# Uses: while loops, input(), dictionaries, functions, sorting

def show_user_ratings(user):
    print(f"\n{user}'s ratings:")
    for movie, rating in ratings[user].items():
        if rating > 0:
            print(f"  {'⭐' * rating}  {movie}")
        else:
            print(f"  [not watched]  {movie}")

def show_all_users():
    print("\nUsers in the system:")
    for i, user in enumerate(ratings.keys(), 1):
        print(f"  {i}. {user}")

def main():
    print("=" * 45)
    print("   MOVIE RECOMMENDATION SYSTEM")
    print("   Collaborative + Content-Based")
    print("=" * 45)

    while True:
        print("\n── MAIN MENU ──────────────────────────")
        print("  1. See recommendations for a user")
        print("  2. See how similar two users are")
        print("  3. View a user's ratings")
        print("  4. Add your own rating")
        print("  5. Quit")
        print("───────────────────────────────────────")

        choice = input("Choose (1-5): ").strip()

        # ── Option 1: Recommendations ────────────────────────────────────
        if choice == "1":
            show_all_users()
            user = input("\nEnter username exactly: ").strip()

            if user not in ratings:
                print("  ⚠  User not found!")
                continue

            show_user_ratings(user)

            print(f"\n── Collaborative Filtering (users like {user}) ──")
            collab = recommend_movies(user, ratings)
            if collab:
                for i, (movie, score) in enumerate(collab, 1):
                    print(f"  {i}. {movie:<15}  predicted: {score:.2f} ⭐")
            else:
                print("  No new recommendations (watched everything!)")

            print(f"\n── Content-Based Filtering (genres {user} likes) ──")
            content = content_recommend(user, ratings, movie_features)
            if content:
                for i, (movie, score) in enumerate(content, 1):
                    print(f"  {i}. {movie:<15}  genre match: {score}")
            else:
                print("  No content recommendations available.")

        # ── Option 2: Compare two users ──────────────────────────────────
        elif choice == "2":
            show_all_users()
            u1 = input("\nFirst user: ").strip()
            u2 = input("Second user: ").strip()

            if u1 not in ratings or u2 not in ratings:
                print("  ⚠  One or both users not found!")
                continue

            score = cosine_similarity(ratings[u1], ratings[u2])
            bar   = "█" * int(score * 30)
            print(f"\n  {u1} vs {u2}")
            print(f"  Similarity: {score:.3f}  {bar}")

            if score > 0.9:
                print("  → Almost identical taste! 🎯")
            elif score > 0.7:
                print("  → Pretty similar taste 👍")
            elif score > 0.4:
                print("  → Some things in common 🤔")
            else:
                print("  → Very different taste 🙅")

        # ── Option 3: View ratings ───────────────────────────────────────
        elif choice == "3":
            show_all_users()
            user = input("\nEnter username: ").strip()
            if user not in ratings:
                print("  ⚠  User not found!")
            else:
                show_user_ratings(user)

        # ── Option 4: Add a rating ───────────────────────────────────────
        elif choice == "4":
            show_all_users()
            user = input("\nEnter username: ").strip()
            if user not in ratings:
                print("  ⚠  User not found!")
                continue

            print("\nMovies:")
            for i, movie in enumerate(all_movies, 1):
                current = ratings[user][movie]
                status  = f"your rating: {current}" if current > 0 else "not watched"
                print(f"  {i}. {movie:<15} ({status})")

            movie = input("\nEnter movie name exactly: ").strip()
            if movie not in all_movies:
                print("  ⚠  Movie not found!")
                continue

            try:
                new_rating = int(input("Enter rating (1-5): "))
                if new_rating not in range(1, 6):
                    print("  ⚠  Rating must be 1 to 5.")
                else:
                    ratings[user][movie] = new_rating
                    print(f"  ✅  Saved! {user} rated {movie}: {'⭐' * new_rating}")
            except ValueError:
                print("  ⚠  Enter a number please.")

        # ── Option 5: Quit ───────────────────────────────────────────────
        elif choice == "5":
            print("\nThanks for using the recommender! Keep learning Python 🐍\n")
            break

        else:
            print("  ⚠  Enter 1, 2, 3, 4, or 5 only.")

# ▶ START
main()